In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from modelSGNS import OnlyOneEmb, SGNS_OneEmbWeighted, SkipGramModel, SGNS_Weighted
from data.pipData import pipe_data, prepare_data, prepare_data_with_intonation, separate_text_intonation
import tool

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import skew

from collections import Counter
import itertools

from dataSet import W2V_weighted_DataSet, W2V_weighted_DataSet_v2, dataset_weighted, SGNS_store_DataSet

import random

from typing import Callable, List, Type

[nltk_data] Downloading package punkt_tab to /home/pe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/pe/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Fct

In [3]:
import os

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    # For absolute reproducibility (may slow down training slightly):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
def createEmbeddingWithSeed(seed:list[int],
                            dataset:Dataset,
                            model:nn.Module,
                            re_init_fct:Callable,
                            get_embedding:Callable,
                            optimizer_cls:type[torch.optim.Optimizer],
                            lr:float, nb_epoch:int,
                            path_to_save:str,
                            words:list[str],
                            device: str = "cpu",
                            batch_size: int = 16,
                            num_workers: int = 0
                        ):
    model.to(device)
    loss_by_seed: List[List[float]] = []
    
    for s in seed:
        set_all_seeds(s)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        
        model = re_init_fct(model)
        optimizer = optimizer_cls(model.parameters(), lr=lr)
        
        loss_by_epoch:list[float] = []
        for epoch in range(nb_epoch):
            loss_in_epoch:list[float] = []
            for sentence_nb, data in enumerate(loader):
                if isinstance(data, (list, tuple)):
                    data = [d.to(device) for d in data]
                else:
                    data = data.to(device)
                    
                optimizer.zero_grad()
                loss:torch.Tensor = model(data)
                loss.backward()
                optimizer.step()
                loss_in_epoch.append(loss.detach().cpu().item())
            loss_by_epoch.append(np.mean(loss_in_epoch))
        
        vectors = get_embedding(model)
        model_name = type(model).__name__
        np.savez(
            f'{path_to_save}/seed_{s}_{model_name}_.npz', 
            vectors=vectors,
            words=words
        )
        loss_by_seed.append(loss_by_epoch)
        print(f"For the seed {s} we have loss {loss_by_epoch}")
        
    
    return loss_by_seed

# Data

In [5]:
corpus = ["GNG"]
all_data_path = {
    "GNG": "./data/GoodNightGorilla_Intonation.txt",
    "IWW": "./data/IwentWalking_intonation.txt",
    "TSGE": "./data/TheShadowGreatEscape_intonation.txt",
    "TRDP": "./data/TheRainyDayParade_intonation.txt",
    "TGTP": "./data/TheGreatToyParade_intonation.txt",
    "TGTR": "./data/TheGreatToyRescue_intonation.txt",
}

In [6]:
texts, intonations = [], []
for book in corpus:
    path = all_data_path[book]
    data = prepare_data_with_intonation(
        file_path=path,
        language='english',
        remove_accent=True,
        remove_punct=True,
        keep_apostrophes=False,
        contraction_map={
            "that's" : "thatis",
            "it's" : "itis",
            "don't": "donot",
            "doesn't": "doesnot",
            "you're": "youre"
        },
        stop_words=["s", "n't"],
        break_line=False
    )

    _texts, _intonations = separate_text_intonation(data)
    intonations.extend(_intonations)
    texts.extend(_texts)

# Norm
norm01 : range_norm = 1.9 and center_norm = 1.

norm02 : range_norm = 1.75 and center_norm = 1.

norm03 : range_norm = 1.5 and center_norm = 1.

norm04 : binary : 0.1 - 1.9, threshold=means


In [ ]:
range_norm = 1.9
center_norm = 1.
print(intonations)
into = tool.normalize_range_center(intonations, range_normalize=range_norm, center=center_norm)
# into = tool.make_binary(intonations, threshold=None, zero_value=0.1, one_value=1.9)

dataset:dataset_weighted = dataset_weighted(sentences=texts,
                                intonations=into, nb_neg=10, window_size=6)

print(dataset.vocab)


[[2, 0, 0, 0, 5, 0, 0, 0, 3, 5, 0, 2, 0, 0, 4, 5, 2, 0, 0, 2, 0, 0, 3, 0, 2, 4, 4, 5, 2, 0, 2, 4, 4], [3, 0, 3, 2, 3, 0, 0, 4, 0, 0, 4, 5, 0, 0, 3, 2, 0, 3, 0, 5, 0, 4, 4, 0, 3, 0, 5, 0, 0, 0, 3, 3, 5, 5], [2, 2, 0, 0, 0, 0, 0, 3, 5, 5, 5, 0, 4, 0, 3, 2], [2, 0, 0, 3, 3, 0, 0, 0, 5, 0, 0, 0, 0, 3, 5, 0, 3, 0, 0, 0, 0, 2, 0, 2, 2, 0, 0, 0, 2, 0, 4, 0, 4], [0, 2, 0, 0, 3, 2, 3, 2, 0, 0, 0, 5, 2, 0, 0, 3, 2], [3, 0, 3, 2, 0, 2], [0, 5, 0, 5, 0, 4, 2, 0, 5, 0, 3, 0, 4, 5], [0, 0, 5, 0, 3, 2, 0, 0, 5, 0, 0, 2, 0, 5, 0, 0, 3, 2, 0, 0, 2], [0, 5, 0, 3, 2, 0, 0, 5, 4, 4, 0, 0, 2, 0, 4, 2, 0, 5, 0, 0, 2, 0, 4, 0, 3, 0, 0, 4, 0, 2, 0, 5, 5, 5], [4, 4, 2, 2, 0, 0, 3, 4, 5, 0, 3, 4, 0, 3, 2, 5], [0, 2, 0, 0, 2, 0, 0, 3, 5, 2, 0, 0, 3, 0, 3, 0, 0, 3, 0, 3, 3, 5, 4, 4, 4], [2, 0, 5, 0, 4, 0, 0, 3, 0, 2, 0, 4, 0, 5, 5, 5], [0, 2, 0, 0, 3, 5, 0, 0, 2, 0, 5, 2, 0, 0, 2, 0, 3, 0, 0, 3, 4, 4, 2, 0, 0, 3, 2, 0, 0], [3, 3, 2, 0, 2, 0, 0, 0, 5, 2, 0, 0, 3, 3, 4, 2, 0, 2, 0, 4, 3, 5], [2, 0, 0, 4, 2, 0, 0, 3

# My method

In [16]:
model = SGNS_OneEmbWeighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_OneEmbWeighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_OneEmbWeighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save=f"embedding/{"_".join(corpus)}/norm01",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(5.002409677400516), np.float64(4.939467102768662), np.float64(4.924994615858435), np.float64(4.903928058113111), np.float64(4.907705831992031), np.float64(4.898289399264123), np.float64(4.89851695930019), np.float64(4.885744068402541), np.float64(4.881633346308903), np.float64(4.886369643627234), np.float64(4.892596511493589), np.float64(4.8783928087260335), np.float64(4.886866877585725), np.float64(4.885284206606794), np.float64(4.883884648364242)]
For the seed 1 we have loss [np.float64(4.996177366383494), np.float64(4.931746169382188), np.float64(4.923623240468464), np.float64(4.910871943107204), np.float64(4.903356830817375), np.float64(4.892614028092464), np.float64(4.898654482507989), np.float64(4.892476241635024), np.float64(4.886018093514503), np.float64(4.891766123000491), np.float64(4.8903605646243244), np.float64(4.8786995999395195), np.float64(4.876989747183894), np.float64(4.887143478112944), np.float64(4.884566489969846)]
For the se

[[np.float64(5.002409677400516),
  np.float64(4.939467102768662),
  np.float64(4.924994615858435),
  np.float64(4.903928058113111),
  np.float64(4.907705831992031),
  np.float64(4.898289399264123),
  np.float64(4.89851695930019),
  np.float64(4.885744068402541),
  np.float64(4.881633346308903),
  np.float64(4.886369643627234),
  np.float64(4.892596511493589),
  np.float64(4.8783928087260335),
  np.float64(4.886866877585725),
  np.float64(4.885284206606794),
  np.float64(4.883884648364242)],
 [np.float64(4.996177366383494),
  np.float64(4.931746169382188),
  np.float64(4.923623240468464),
  np.float64(4.910871943107204),
  np.float64(4.903356830817375),
  np.float64(4.892614028092464),
  np.float64(4.898654482507989),
  np.float64(4.892476241635024),
  np.float64(4.886018093514503),
  np.float64(4.891766123000491),
  np.float64(4.8903605646243244),
  np.float64(4.8786995999395195),
  np.float64(4.876989747183894),
  np.float64(4.887143478112944),
  np.float64(4.884566489969846)],
 [np.f

# Two embedding weighted

In [ ]:
model = SGNS_Weighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_Weighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_Weighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save=f"embedding/{"_".join(corpus)}/norm01",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

# OnlyOneEmb

In [ ]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = OnlyOneEmb(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_one_emb(model:OnlyOneEmb):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_one_emb(model:OnlyOneEmb):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10), 
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_one_emb,
                        get_embedding=get_embedding_w2v_one_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save=f"embedding/{"_".join(corpus)}/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

# SkipGramModel

In [ ]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = SkipGramModel(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_two_emb(model:SkipGramModel):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_two_emb(model:SkipGramModel):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = range(0, 10),
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_two_emb,
                        get_embedding=get_embedding_w2v_two_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=15,
                        path_to_save=f"embedding/{"_".join(corpus)}/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

# Step 

## Data

In [3]:
step  = 250
corpus = ["GNG"]
all_data_path = {
    "GNG": "./data/GoodNightGorilla_Intonation.txt",
    "IWW": "./data/IwentWalking_intonation.txt",
    "TSGE": "./data/TheShadowGreatEscape_intonation.txt",
    "TRDP": "./data/TheRainyDayParade_intonation.txt",
    "TGTP": "./data/TheGreatToyParade_intonation.txt",
    "TGTR": "./data/TheGreatToyRescue_intonation.txt",
}

In [4]:
texts, intonations = [], []
for book in corpus:
    path = all_data_path[book]
    data = prepare_data_with_intonation(
        file_path=path,
        language='english',
        remove_accent=True,
        remove_punct=True,
        keep_apostrophes=False,
        contraction_map={
            "that's" : "thatis",
            "it's" : "itis",
            "don't": "donot",
            "doesn't": "doesnot",
            "you're": "youre"
        },
        stop_words=["s", "n't"],
        break_line=False
    )

    _texts, _intonations = separate_text_intonation(data)
    intonations.extend(_intonations)
    texts.extend(_texts)


flat_text = list(itertools.chain(*texts))
print(flat_text)
text_step = []
for index in range(len(flat_text) // 250 - 1):
    text_step.append(flat_text[0:(index + 1)*250])
    print(text_step[-1])

text_step.append(flat_text)

flat_intonation = list(itertools.chain(*intonations))
print(flat_intonation)

intonation_step = []
for index in range(len(flat_intonation) // 250 - 1):
    intonation_step.append(flat_intonation[0:(index + 1)*250])
    print(intonation_step[-1])

intonation_step.append(flat_intonation)
print(len(intonation_step[-1]))

['look', 'there', 'is', 'the', 'zookeeper', 'he', 'has', 'a', 'big', 'flashlight', 'to', 'see', 'in', 'the', 'dark', 'click', 'what', 'is', 'he', 'saying', 'to', 'the', 'animal', 'he', 'says', 'good', 'night', 'gorilla', 'can', 'you', 'say', 'good', 'night', 'oh', 'my', 'goodness', 'look', 'closer', 'is', 'the', 'gorilla', 'going', 'to', 'sleep', 'no', 'he', 'is', 'reaching', 'out', 'and', 'taking', 'the', 'keys', 'that', 'sneaky', 'gorilla', 'is', 'stealing', 'the', 'keys', 'right', 'off', 'the', 'zookeeper', 'belt', 'jingle', 'jangle', 'who', 'sees', 'him', 'doing', 'it', 'it', 'the', 'little', 'mouse', 'squeak', 'squeak', 'the', 'mouse', 'is', 'watching', 'everything', 'look', 'at', 'the', 'gorilla', 'room', 'he', 'has', 'a', 'bicycle', 'in', 'there', 'and', 'a', 'big', 'tire', 'to', 'swing', 'on', 'but', 'he', 'doesnot', 'want', 'to', 'stay', 'inside', 'does', 'he', 'he', 'wants', 'to', 'follow', 'the', 'zookeeper', 'and', 'what', 'is', 'that', 'pink', 'thing', 'floating', 'up', 't

# Norm and generate embeddings

In [ ]:
range_norm = 1.9
center_norm = 1.
name_norm = "norm01"

for num_sub_corpus in range(len(flat_text)):
    
    print(f"=========== test sub : {num_sub_corpus}/{len(flat_text)} ==========")
    into = tool.normalize_range_center([intonation_step[num_sub_corpus]], range_normalize=range_norm,
                                    center=center_norm)
    # into = tool.make_binary(intonations, threshold=None, zero_value=0.1, one_value=1.9)

    dataset_weight:dataset_weighted = dataset_weighted(sentences=[text_step[num_sub_corpus]],
                                    intonations=into, nb_neg=10, window_size=6)
    
    dataset = SGNS_store_DataSet(sentences=[text_step[num_sub_corpus]], nb_neg=10, power=0.75,
                                subsample_thresh=0, window_size=6)
    
    path_to_save = f"embedding/{"_".join(corpus)}/noNorm/by_step_{step}/{num_sub_corpus}"
    path_to_save_norm = f"embedding/{"_".join(corpus)}/{name_norm}/by_step_{step}/{num_sub_corpus}"
    os.makedirs(path_to_save, exist_ok=True)
    os.makedirs(path_to_save_norm, exist_ok=True)
    
    
    # =================== Model SGNS_OneEmbWeighted =================== 
    model = SGNS_OneEmbWeighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

    def re_init_model_w2v_weighted(model:SGNS_OneEmbWeighted):
        with torch.no_grad():
            model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        return model

    def get_embedding_w2v_weighted(model:SGNS_OneEmbWeighted):
        return model.word_emb.weight.detach().cpu().numpy()

    createEmbeddingWithSeed(seed = range(0, 10),
                            dataset=dataset_weight,
                            model=model,
                            re_init_fct=re_init_model_w2v_weighted,
                            get_embedding=get_embedding_w2v_weighted,
                            optimizer_cls=torch.optim.Adam,
                            lr=0.003, nb_epoch=15,
                            path_to_save=path_to_save_norm,
                            words=list(dataset.encoder.keys()),
                            device="cuda",
                            batch_size= 16,
                            num_workers= 0)
    
    
    # =================== Model SGNS_Weighted =================== 
    model = SGNS_Weighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

    def re_init_model_w2v_weighted(model:SGNS_Weighted):
        with torch.no_grad():
            model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
            model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
        return model

    def get_embedding_w2v_weighted(model:SGNS_Weighted):
        return model.word_emb.weight.detach().cpu().numpy()

    createEmbeddingWithSeed(seed = range(0, 10),
                            dataset=dataset_weight,
                            model=model,
                            re_init_fct=re_init_model_w2v_weighted,
                            get_embedding=get_embedding_w2v_weighted,
                            optimizer_cls=torch.optim.Adam,
                            lr=0.003, nb_epoch=15,
                            path_to_save=path_to_save_norm,
                            words=list(dataset.encoder.keys()),
                            device="cuda",
                            batch_size= 16,
                            num_workers= 0)
    
    
    # =================== Model OnlyOneEmb =================== 
    model = OnlyOneEmb(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

    def re_init_model_w2v_one_emb(model:OnlyOneEmb):
        with torch.no_grad():
            model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        return model

    def get_embedding_w2v_one_emb(model:OnlyOneEmb):
        return model.word_emb.weight.detach().cpu().numpy()

    createEmbeddingWithSeed(seed = range(0, 10), 
                            dataset=dataset,
                            model=model,
                            re_init_fct=re_init_model_w2v_one_emb,
                            get_embedding=get_embedding_w2v_one_emb,
                            optimizer_cls=torch.optim.Adam,
                            lr=0.003, nb_epoch=15,
                            path_to_save=path_to_save,
                            words=list(dataset.encoder.keys()),
                            device="cuda",
                            batch_size= 16,
                            num_workers= 0)


    # =================== Model SkipGramModel =================== 
    model = SkipGramModel(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)
    
    def re_init_model_w2v_two_emb(model:SkipGramModel):
        with torch.no_grad():
            model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
            model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
        return model

    def get_embedding_w2v_two_emb(model:SkipGramModel):
        return model.word_emb.weight.detach().cpu().numpy()

    createEmbeddingWithSeed(seed = range(0, 10),
                            dataset=dataset,
                            model=model,
                            re_init_fct=re_init_model_w2v_two_emb,
                            get_embedding=get_embedding_w2v_two_emb,
                            optimizer_cls=torch.optim.Adam,
                            lr=0.003, nb_epoch=15,
                            path_to_save=path_to_save,
                            words=list(dataset.encoder.keys()),
                            device="cuda",
                            batch_size= 16,
                            num_workers= 0)

    

[2, 0, 0, 0, 5, 0, 0, 0, 3, 5, 0, 2, 0, 0, 4, 5, 2, 0, 0, 2, 0, 0, 3, 0, 2, 4, 4, 5, 2, 0, 2, 4, 4, 3, 0, 3, 2, 3, 0, 0, 4, 0, 0, 4, 5, 0, 0, 3, 2, 0, 3, 0, 5, 0, 4, 4, 0, 3, 0, 5, 0, 0, 0, 3, 3, 5, 5, 2, 2, 0, 0, 0, 0, 0, 3, 5, 5, 5, 0, 4, 0, 3, 2, 2, 0, 0, 3, 3, 0, 0, 0, 5, 0, 0, 0, 0, 3, 5, 0, 3, 0, 0, 0, 0, 2, 0, 2, 2, 0, 0, 0, 2, 0, 4, 0, 4, 0, 2, 0, 0, 3, 2, 3, 2, 0, 0, 0, 5, 2, 0, 0, 3, 2, 3, 0, 3, 2, 0, 2, 0, 5, 0, 5, 0, 4, 2, 0, 5, 0, 3, 0, 4, 5, 0, 0, 5, 0, 3, 2, 0, 0, 5, 0, 0, 2, 0, 5, 0, 0, 3, 2, 0, 0, 2, 0, 5, 0, 3, 2, 0, 0, 5, 4, 4, 0, 0, 2, 0, 4, 2, 0, 5, 0, 0, 2, 0, 4, 0, 3, 0, 0, 4, 0, 2, 0, 5, 5, 5, 4, 4, 2, 2, 0, 0, 3, 4, 5, 0, 3, 4, 0, 3, 2, 5, 0, 2, 0, 0, 2, 0, 0, 3, 5, 2, 0, 0, 3, 0, 3, 0, 0, 3, 0, 3, 3, 5, 4, 4, 4, 2]
For the seed 0 we have loss [np.float64(5.588692447301503), np.float64(5.530399055738707), np.float64(5.450205978187355), np.float64(5.435367852288324), np.float64(5.403950255625957), np.float64(5.400835417412423), np.float64(5.381061184084094), np.

IndexError: list index out of range